# CardioTwin Model Optimization

IIT Multimodal AI Hackathon 2026 Track A.

This notebook performs target-specific, leakage-safe optimization for CAD, LAD, LCX, and RCA. It uses repeated stratified CV, training-fold-only preprocessing/feature selection/class weighting/calibration, and exports optimization artifacts. No SMOTE, synthetic augmentation, full-dataset feature selection, target-as-predictor modeling, threshold tuning on final-test labels, or production model saving is performed.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    for maybe_ml in [candidate, candidate / "ml"]:
        if (maybe_ml / "src" / "models" / "optimization.py").exists():
            if str(maybe_ml) not in sys.path:
                sys.path.insert(0, str(maybe_ml))
            break
    else:
        continue
    break

from src.models.optimization import (
    FORBIDDEN_COLUMNS,
    TARGETS,
    find_project_root,
    load_inputs,
    run_optimization,
)

PROJECT_ROOT = find_project_root()
X, y, feature_metadata, numeric_features, categorical_features, baseline_summary, baseline_oof, baseline_overfit = load_inputs(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"X shape: {X.shape}")
print(f"Targets: {list(y.columns)}")
print(f"Numeric predictors: {len(numeric_features)}")
print(f"Categorical/binary predictors: {len(categorical_features)}")
assert FORBIDDEN_COLUMNS.isdisjoint(X.columns), "Forbidden leakage columns found in X"
assert X.index.equals(y.index), "X/y index mismatch"
for target in TARGETS:
    assert set(y[target].unique()).issubset({0, 1}), f"Non-binary target: {target}"
print("Safety checks passed.")

Project root: C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai
X shape: (303, 55)
Targets: ['cad', 'lad', 'lcx', 'rca']
Numeric predictors: 21
Categorical/binary predictors: 34
Safety checks passed.


## Execute Optimization

The reusable optimization harness evaluates target-specific promising candidates, ablations, calibration strategies, thresholds, simple OOF soft-voting ensembles, and baseline-vs-optimized comparisons. All fold-dependent operations are fitted only on training folds.

In [2]:
results = run_optimization(PROJECT_ROOT)
report = results["report"]
print(json.dumps(report, indent=2, default=str))

C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


...

C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\samri\cod\git\IIT_Mandi\cardio-3d-ai\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Evaluated 4/4: rca logreg_l2_C0.1_cwbalanced_kall calibration=isotonic


{
  "execution_seconds": 645.1324444999918,
  "baseline_vs_optimized": [
    {
      "target": "cad",
      "baseline_model": "logreg_l2",
      "optimized_model": "logreg_l2_C0.1_cwbalanced_kall__drop_lymph",
      "baseline_roc_auc": 0.918353300446324,
      "optimized_roc_auc": 0.9237534027000511,
      "delta_roc_auc": 0.005400102253727113,
      "baseline_pr_auc": 0.9658656508441916,
      "optimized_pr_auc": 0.9681461143150705,
      "delta_pr_auc": 0.0022804634708788463,
      "baseline_f1": 0.893906573668001,
      "optimized_f1": 0.8920922876852998,
      "delta_f1": -0.0018142859827011693,
      "baseline_balanced_accuracy": 0.8202844449971675,
      "optimized_balanced_accuracy": 0.8519881440948472,
      "delta_balanced_accuracy": 0.03170369909767978,
      "baseline_brier": 0.1086689911447147,
      "optimized_brier": 0.11095210314606055,
      "delta_brier": 0.002283112001345844,
      "baseline_roc_auc_std": 0.0338984003042368,
      "optimized_roc_auc_std": 0.0323822417

## Export Verification

In [3]:
expected = [
    "hyperparameter_trials.csv",
    "optimized_cv_metrics.csv",
    "optimized_summary.csv",
    "optimized_oof_predictions.csv",
    "feature_strategy_comparison.csv",
    "correlated_feature_ablation.csv",
    "echo_feature_ablation.csv",
    "calibration_comparison.csv",
    "threshold_analysis.csv",
    "ensemble_comparison.csv",
    "baseline_vs_optimized.csv",
    "optimization_stability.csv",
]
out_dir = results["output_dir"]
for name in expected:
    path = out_dir / name
    assert path.exists(), f"Missing export: {path}"
    print(name, pd.read_csv(path).shape)
assert not (out_dir / "production_model.joblib").exists()
print("All exports verified. No production models saved.")

hyperparameter_trials.csv (24, 13)
optimized_cv_metrics.csv (1000, 26)
optimized_summary.csv (40, 28)


optimized_oof_predictions.csv (60600, 13)
feature_strategy_comparison.csv (16, 28)
correlated_feature_ablation.csv (12, 28)
echo_feature_ablation.csv (8, 28)
calibration_comparison.csv (15, 7)
threshold_analysis.csv (255, 18)
ensemble_comparison.csv (36, 18)
baseline_vs_optimized.csv (4, 21)
optimization_stability.csv (16, 11)
All exports verified. No production models saved.


## Final Recommendations

In [4]:
optimized_summary = pd.read_csv(out_dir / "optimized_summary.csv")
baseline_vs_optimized = pd.read_csv(out_dir / "baseline_vs_optimized.csv")
calibration = pd.read_csv(out_dir / "calibration_comparison.csv")
thresholds = pd.read_csv(out_dir / "threshold_analysis.csv")
stability = pd.read_csv(out_dir / "optimization_stability.csv")

print("Baseline vs optimized:")
display(baseline_vs_optimized)

print("Top optimized candidates per target:")
display(optimized_summary.groupby("target").head(5))

print("Calibration comparison, top rows per target by Brier/ECE:")
display(calibration.groupby("target").head(5))

print("Candidate threshold trade-offs, top F1 rows per target:")
display(thresholds.sort_values(["target", "f1", "balanced_accuracy"], ascending=[True, False, False]).groupby("target").head(5))

print("Overfitting/stability warnings:")
display(stability[stability["overfitting_flag"] == True])

print("All reported optimized validation predictions were generated out-of-fold with preprocessing, feature selection, class weighting, calibration, and fitting performed only on training folds.")

Baseline vs optimized:


,target,baseline_model,optimized_model,baseline_roc_auc,optimized_roc_auc,delta_roc_auc,baseline_pr_auc,optimized_pr_auc,delta_pr_auc,baseline_f1,...,delta_f1,baseline_balanced_accuracy,optimized_balanced_accuracy,delta_balanced_accuracy,baseline_brier,optimized_brier,delta_brier,baseline_roc_auc_std,optimized_roc_auc_std,delta_roc_auc_std
0,cad,logreg_l2,logreg_l2_C0.1_cwbalanced_kall__drop_lymph,0.918353,0.923753,0.005400,0.965866,0.968146,0.002280,0.893907,...,-0.001814,0.820284,0.851988,0.031704,0.108669,0.110952,0.002283,0.033898,0.032382,-0.001516
1,lad,extra_trees,random_forest_d4_leaf5_cwbalanced,0.821955,0.836433,0.014478,0.862451,0.870834,0.008383,0.807111,...,-0.002703,0.740959,0.765952,0.024993,0.175027,0.174347,-0.000680,0.055593,0.049939,-0.005654
2,lcx,xgboost,logreg_elasticnet_C0.1_cwbalanced_k30__drop_neut,0.694674,0.728496,0.033823,0.574854,0.612714,0.037860,0.000000,...,0.608258,0.500000,0.669510,0.169510,0.224017,0.212956,-0.011061,0.074745,0.072276,-0.002469
3,rca,logreg_l2_balanced,logreg_l2_C0.1_cwbalanced_kall__no_echo,0.712632,0.737030,0.024397,0.604957,0.622098,0.017141,0.574624,...,0.033871,0.650762,0.677148,0.026386,0.230063,0.207485,-0.022579,0.060217,0.052868,-0.007349


Top optimized candidates per target:


,target,model,family,feature_strategy,selection_k,class_weight_mode,calibration,n_folds,accuracy_mean,accuracy_std,...,pr_auc_average_precision_mean,pr_auc_average_precision_std,specificity_mean,specificity_std,balanced_accuracy_mean,balanced_accuracy_std,mcc_mean,mcc_std,brier_score_mean,brier_score_std
0,cad,logreg_l2_C0.1_cwbalanced_kall__drop_lymph,logreg,drop_lymph,all,balanced,none,25,0.853410,0.042779,...,0.968146,0.014593,0.848627,0.085201,0.851988,0.048394,0.672855,0.093778,0.110952,0.022778
1,cad,logreg_l2_C0.1_cwbalanced_kall__drop_neut,logreg,drop_neut,all,balanced,none,25,0.851443,0.041879,...,0.967938,0.014347,0.844183,0.090288,0.849301,0.048982,0.667931,0.092712,0.111220,0.022894
2,cad,logreg_l2_C0.1_cwbalanced_kall,logreg,all,all,balanced,none,25,0.854066,0.042099,...,0.967754,0.014849,0.848627,0.085201,0.852453,0.047885,0.673988,0.092547,0.111428,0.023066
3,cad,logreg_l2_C0.1_cwbalanced_kall__all,logreg,all,all,balanced,none,25,0.854066,0.042099,...,0.967754,0.014849,0.848627,0.085201,0.852453,0.047885,0.673988,0.092547,0.111428,0.023066
4,cad,logreg_l2_C0.1_cwbalanced_kall,logreg,all,all,balanced,sigmoid,25,0.862732,0.045903,...,0.967460,0.015385,0.720392,0.107930,0.820355,0.059912,0.660816,0.116902,0.099972,0.022345
10,lad,random_forest_d4_leaf5_cwbalanced,random_forest,all,all,balanced,sigmoid,25,0.775694,0.049081,...,0.876132,0.047699,0.673354,0.095547,0.761185,0.051089,0.538659,0.099997,0.159714,0.023938
11,lad,random_forest_d4_leaf5_cwbalanced,random_forest,all,all,balanced,none,25,0.772350,0.045836,...,0.870834,0.041226,0.727015,0.079306,0.765952,0.046583,0.535155,0.092004,0.174347,0.014285
12,lad,random_forest_d4_leaf5_cwbalanced__all,random_forest,all,all,balanced,none,25,0.772350,0.045836,...,0.870834,0.041226,0.727015,0.079306,0.765952,0.046583,0.535155,0.092004,0.174347,0.014285
13,lad,random_forest_d4_leaf5_cwbalanced,random_forest,all,all,balanced,isotonic,25,0.775639,0.050207,...,0.866409,0.049492,0.662215,0.097490,0.759536,0.052058,0.538775,0.103207,0.163374,0.028101
14,lad,random_forest_d4_leaf5_cwbalanced__drop_neut,random_forest,drop_neut,all,balanced,none,25,0.766546,0.044393,...,0.869629,0.053033,0.711631,0.112112,0.758895,0.049626,0.524049,0.094218,0.176229,0.014660


Calibration comparison, top rows per target by Brier/ECE:


...

Candidate threshold trade-offs, top F1 rows per target:


,target,model,calibration,threshold,accuracy,precision,recall_sensitivity,f1,roc_auc,pr_auc_average_precision,specificity,balanced_accuracy,mcc,brier_score,tn,fp,fn,tp
21,cad,logreg_l2_C0.1_cwbalanced_kall,none,0.30,0.864026,0.884683,0.930556,0.907040,0.922401,0.966801,0.698851,0.814703,0.657474,0.111424,304,131,75,1005
41,cad,logreg_l2_C0.1_cwbalanced_kall,sigmoid,0.45,0.862706,0.881786,0.932407,0.906391,0.921043,0.966320,0.689655,0.811031,0.653286,0.099981,300,135,73,1007
42,cad,logreg_l2_C0.1_cwbalanced_kall,sigmoid,0.50,0.862706,0.890681,0.920370,0.905282,0.921043,0.966320,0.719540,0.819955,0.657290,0.099981,313,122,86,994
57,cad,random_forest_d4_leaf5_cwbalanced,none,0.40,0.863366,0.900092,0.909259,0.904652,0.908863,0.961156,0.749425,0.829342,0.663802,0.129935,326,109,98,982
22,cad,logreg_l2_C0.1_cwbalanced_kall,none,0.35,0.861386,0.891892,0.916667,0.904110,0.922401,0.966801,0.724138,0.820402,0.655078,0.111424,315,120,90,990
108,lad,random_forest_d4_leaf5_cwbalanced,sigmoid,0.40,0.783498,0.762984,0.912994,0.831276,0.832826,0.855222,0.601587,0.757291,0.552924,0.159770,379,251,77,808
109,lad,random_forest_d4_leaf5_cwbalanced,sigmoid,0.45,0.781518,0.774257,0.883616,0.825330,0.832826,0.855222,0.638095,0.760856,0.545463,0.159770,402,228,103,782
107,lad,random_forest_d4_leaf5_cwbalanced,sigmoid,0.35,0.768317,0.740975,0.927684,0.823884,0.832826,0.855222,0.544444,0.736064,0.524970,0.159770,343,287,64,821
75,lad,random_forest_d4_leaf5_cwbalanced,isotonic,0.45,0.778218,0.774226,0.875706,0.821845,0.824746,0.847038,0.641270,0.758488,0.538163,0.163433,404,226,110,775
74,lad,random_forest_d4_leaf5_cwbalanced,isotonic,0.40,0.772277,0.758621,0.894915,0.821151,0.824746,0.847038,0.600000,0.747458,0.527002,0.163433,378,252,93,792


Overfitting/stability warnings:


,target,model,train_roc_auc_mean,valid_roc_auc_mean,roc_auc_gap_train_minus_valid,train_f1_mean,valid_f1_mean,f1_gap_train_minus_valid,train_brier_score_mean,valid_brier_score_mean,overfitting_flag
0,lcx,xgboost_d2_lr0.05_cwscale_pos_weight,0.863899,0.706700,0.157199,0.746192,0.603584,0.142608,0.192252,0.220789,True
1,rca,logreg_l2_C0.1_cwbalanced_kall,0.856916,0.725708,0.131208,0.661186,0.485604,0.175581,0.160995,0.206847,True


All reported optimized validation predictions were generated out-of-fold with preprocessing, feature selection, class weighting, calibration, and fitting performed only on training folds.
